# Weather Data Diagnosis 與 Training Workflow

這份 notebook 使用 MySQL 中的真實小時天氣資料。目標是先判斷資料 behavior 較接近哪一種類型，再選擇對應的 simulation case。整體 workflow 使用最近 7 天資料作為 rolling training window，並建立未來 24 小時的 forecasting setup。

## 1. 載入 Packages 與 Project Settings

這個 cell 會載入基本的 scientific Python stack。Machine Learning models 使用 `scikit-learn`，目前環境已經可以使用。如果 VS Code 選到的 kernel 可以 import `PyKrige`，就會使用 Ordinary Kriging；否則 notebook 會使用 inverse distance weighting (IDW) 作為 fallback。

In [1]:
from pathlib import Path
import sys
import importlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

try:
    from pykrige.ok import OrdinaryKriging
except ImportError:
    OrdinaryKriging = None

ROOT = Path.cwd().parent if Path.cwd().name == 'main' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))

import config as config_module
import db as db_module
importlib.reload(config_module)
importlib.reload(db_module)
settings = config_module.settings
read_observations = db_module.read_observations

sns.set_theme(style='whitegrid')
TRAINING_HOURS = 168
FORECAST_HOURS = 24
VARIABLE = settings.forecast_variable
safe_db_settings = dict(config_module.db_settings)
safe_db_settings['password_set'] = bool(safe_db_settings.pop('password'))
print({'variable': VARIABLE, 'training_hours': TRAINING_HOURS, 'forecast_hours': FORECAST_HOURS})
print('db_settings:', safe_db_settings)

{'variable': 'temperature', 'training_hours': 168, 'forecast_hours': 24}
db_settings: {'server': '127.0.0.1', 'port': 3306, 'user': 'sammy', 'database': 'homework', 'charset': 'utf8', 'password_set': False}


## 2. 從 MySQL 讀取最近 7 天資料

project database 會把真實 observation 存在 `observations_hourly`，測站座標存在 `stations`。這個 cell 會針對一個 weather variable 讀取最近 168 小時資料。如果要診斷 humidity、wind speed 或 pressure，可以修改 `VARIABLE`。

In [3]:
df = read_observations(VARIABLE, limit_hours=TRAINING_HOURS)
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.dropna(subset=['lat', 'lon', 'datetime', 'value']).sort_values(['datetime', 'station_id'])
display(df.head())
print('rows:', len(df), 'stations:', df['station_id'].nunique(), 'time range:', df['datetime'].min(), 'to', df['datetime'].max())

ProgrammingError: 1045 (28000): Access denied for user 'root'@'localhost' (using password: NO)

## 3. 診斷 Data Type

paper 比較了不同 spatial data behavior，例如 Gaussian、complex/non-stationary、trend 與 lognormal cases。對 weather data 來說，可以用 skewness、log-skewness、temporal trend strength 與 spatial nonlinearity 來判斷最接近的 behavior。這不是嚴格定理，而是用來選擇 simulation case 的 practical rule。

In [ ]:
def diagnose_data_type(data):
    x = data['value'].astype(float).to_numpy()
    x_skew = float(skew(x, nan_policy='omit'))
    x_kurt = float(kurtosis(x, nan_policy='omit'))
    positive = x[x > 0]
    log_skew = float(skew(np.log1p(positive), nan_policy='omit')) if len(positive) else np.nan

    time_index = (data['datetime'] - data['datetime'].min()).dt.total_seconds().to_numpy().reshape(-1, 1)
    trend_model = LinearRegression().fit(time_index, x)
    trend_r2 = float(trend_model.score(time_index, x))

    latest = data[data['datetime'] == data['datetime'].max()].copy()
    spatial_r2 = np.nan
    if len(latest) >= 8:
        X = latest[['lon', 'lat']].to_numpy()
        y = latest['value'].to_numpy()
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42)
        rf = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=2)
        rf.fit(X_train, y_train)
        spatial_r2 = float(r2_score(y_test, rf.predict(X_test)))

    if abs(x_skew) < 0.5 and trend_r2 < 0.2:
        case = 'gaussian'
    elif x_skew > 1.0 and abs(log_skew) < abs(x_skew):
        case = 'skewed_lognormal'
    elif trend_r2 >= 0.25:
        case = 'trend'
    else:
        case = 'non_stationary'

    return {
        'case': case,
        'skewness': x_skew,
        'kurtosis': x_kurt,
        'log1p_skewness_positive_values': log_skew,
        'temporal_trend_r2': trend_r2,
        'latest_hour_spatial_rf_r2': spatial_r2,
    }

diagnosis = diagnose_data_type(df)
print(json.dumps(diagnosis, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['value'], kde=True, ax=axes[0])
axes[0].set_title(f'{VARIABLE} distribution')
hourly = df.groupby('datetime')['value'].mean().reset_index()
sns.lineplot(data=hourly, x='datetime', y='value', ax=axes[1])
axes[1].set_title('Hourly mean over the 7-day window')
plt.tight_layout()

## 4. Spatial Baseline、Machine Learning 與 Fusion

這一節比較 Ordinary Kriging、Machine Learning 與 weighted Fusion model。如果目前 VS Code kernel 無法 import `PyKrige`，會使用 IDW 作為 fallback，讓實驗仍然可以執行。

In [ ]:
def idw_predict(train_xy, train_y, target_xy, k=8):
    tree = cKDTree(train_xy)
    k = min(k, len(train_xy))
    distances, indexes = tree.query(target_xy, k=k)
    if k == 1:
        distances = distances[:, None]
        indexes = indexes[:, None]
    distances = np.maximum(distances, 1e-9)
    weights = 1.0 / distances**2
    weights = weights / weights.sum(axis=1, keepdims=True)
    pred = np.sum(train_y[indexes] * weights, axis=1)
    var = np.average((train_y[indexes] - pred[:, None])**2, axis=1, weights=weights)
    return pred, var

latest = df[df['datetime'] == df['datetime'].max()].copy()
train_sp, test_sp = train_test_split(latest, test_size=0.35, random_state=42)
X_train = train_sp[['lon', 'lat']].to_numpy(float)
y_train = train_sp['value'].to_numpy(float)
X_test = test_sp[['lon', 'lat']].to_numpy(float)
y_test = test_sp['value'].to_numpy(float)

def kriging_predict(train_xy, train_y, target_xy):
    if OrdinaryKriging is None or len(train_xy) < 4:
        pred, var = idw_predict(train_xy, train_y, target_xy)
        return pred, var, 'IDW fallback'
    try:
        ok = OrdinaryKriging(
            train_xy[:, 0], train_xy[:, 1], train_y,
            variogram_model='spherical', verbose=False, enable_plotting=False
        )
        pred, var = ok.execute('points', target_xy[:, 0], target_xy[:, 1])
        return np.asarray(pred, dtype=float), np.asarray(var, dtype=float), 'OrdinaryKriging'
    except Exception as exc:
        print('PyKrige failed; using IDW fallback:', exc)
        pred, var = idw_predict(train_xy, train_y, target_xy)
        return pred, var, 'IDW fallback'

kr_pred, kr_var, kriging_method = kriging_predict(X_train, y_train, X_test)
print('spatial baseline:', kriging_method)

learners = {
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=300, random_state=42, min_samples_leaf=2),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
}
ml_preds = []
for name, model in learners.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    ml_preds.append(pred)
ml_pred = np.mean(ml_preds, axis=0)

scaled_var = kr_var / max(float(np.max(kr_var)), 1e-9)
w = np.clip(scaled_var, 0, 1)
fusion_pred = w * ml_pred + (1 - w) * kr_pred

def metrics(name, pred):
    return {
        'model': name,
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'MAE': mean_absolute_error(y_test, pred),
        'R2': r2_score(y_test, pred),
    }

results = pd.DataFrame([
    metrics(kriging_method, kr_pred),
    metrics('ML ensemble', ml_pred),
    metrics('Weighted fusion', fusion_pred),
])
display(results)
sns.barplot(data=results, x='model', y='RMSE')
plt.xticks(rotation=20, ha='right')
plt.title('Spatial holdout RMSE comparison')
plt.tight_layout()

## 5. AutoAR-Style Temporal Forecasting

在 temporal part 中，每個 station 可以自動選擇 AR lag。這個輕量版本會對 lagged values 使用 linear regression，並選擇 validation RMSE 最低的 lag。之後 final backend 若需要，可以再替換成 `statsmodels` 的 AutoReg。

In [ ]:
def select_ar_lag(values, max_lag=24):
    values = pd.Series(values).dropna().astype(float).to_numpy()
    if len(values) < 8:
        return 1, np.nan
    best = (1, np.inf)
    for lag in range(1, min(max_lag, len(values)//3) + 1):
        X, y = [], []
        for i in range(lag, len(values)):
            X.append(values[i-lag:i])
            y.append(values[i])
        X, y = np.asarray(X), np.asarray(y)
        split = max(1, int(len(y) * 0.8))
        model = LinearRegression().fit(X[:split], y[:split])
        pred = model.predict(X[split:]) if split < len(y) else model.predict(X[:split])
        truth = y[split:] if split < len(y) else y[:split]
        rmse = mean_squared_error(truth, pred) ** 0.5
        if rmse < best[1]:
            best = (lag, rmse)
    return best

ar_rows = []
for station_id, group in df.groupby('station_id'):
    lag, rmse = select_ar_lag(group.sort_values('datetime')['value'], max_lag=24)
    ar_rows.append({'station_id': station_id, 'selected_lag': lag, 'validation_rmse': rmse})
ar_summary = pd.DataFrame(ar_rows).sort_values('validation_rmse')
display(ar_summary.head(10))
sns.countplot(data=ar_summary, x='selected_lag')
plt.title('Automatically selected AR lag by station')
plt.tight_layout()